# P04B — Saliency y atribución en visión

Esta demo corta cambia el tipo de entrada.

En lugar de una tabla, utilizaremos una imagen y un modelo preentrenado de visión.

Compararemos:

- **Saliency** basada en gradientes;
- **Integrated Gradients**.

La pregunta es:

> ¿Qué regiones de la entrada influyen en el score de una clase concreta?

Una heatmap de atribución no demuestra causalidad ni que el modelo “razone” visualmente como una persona.


## 1. Preparar Captum

La demo funciona en CPU.

Si Colab dispone de GPU, PyTorch la utilizará automáticamente.


In [ ]:
!pip -q install captum==0.9.0


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import urllib.request

from PIL import Image

from torchvision.models import resnet18, ResNet18_Weights
from captum.attr import Saliency, IntegratedGradients

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Dispositivo:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 2. Cargar una imagen y un modelo preentrenado

Usaremos ResNet-18 con pesos ImageNet.

El propio objeto `weights` proporciona:

- preprocesamiento correcto;
- nombres de clases.


In [ ]:
IMAGE_URL = "https://github.com/pytorch/hub/raw/master/images/dog.jpg"
IMAGE_PATH = "/content/p04b_dog.jpg"

urllib.request.urlretrieve(IMAGE_URL, IMAGE_PATH)

image = Image.open(IMAGE_PATH).convert("RGB")

weights = ResNet18_Weights.DEFAULT
model = resnet18(weights=weights).to(device).eval()

preprocess = weights.transforms()
categories = weights.meta["categories"]

input_tensor = preprocess(image).unsqueeze(0).to(device)

plt.figure(figsize=(7, 5))
plt.imshow(image)
plt.axis("off")
plt.title("Imagen de entrada")
plt.show()


## 3. Obtener las clases con mayor probabilidad

Primero vemos qué predice el modelo.

Después podremos explicar:

- la clase principal;
- o una de las alternativas del top-5.

Esto es importante porque una atribución siempre se calcula respecto a una **salida concreta**.


In [ ]:
with torch.no_grad():
    logits = model(input_tensor)
    probabilities = torch.softmax(logits, dim=1)[0]

top_probs, top_idx = torch.topk(probabilities, 5)

top5 = []
for rank, (prob, idx) in enumerate(zip(top_probs.cpu(), top_idx.cpu()), start=1):
    top5.append({
        "rank": rank,
        "clase": categories[int(idx)],
        "probabilidad": float(prob),
        "class_id": int(idx),
    })

import pandas as pd
top5_df = pd.DataFrame(top5)
display(top5_df.style.format({"probabilidad": "{:.3%}"}))


## 4. Elegir la clase que queremos explicar

Prueba a explicar:

- la clase top-1;
- otra clase del top-5.

La heatmap puede cambiar porque estamos preguntando al modelo por una salida diferente.


In [ ]:
TARGET_RANK = 1 #@param {type:"slider", min:1, max:5, step:1}

target_class = int(top5_df.loc[TARGET_RANK - 1, "class_id"])
target_name = top5_df.loc[TARGET_RANK - 1, "clase"]

print("Clase explicada:", target_name)


# 5. Saliency basada en gradientes

Saliency calcula el gradiente de la salida respecto a los píxeles.

Una magnitud grande indica que un pequeño cambio local en esa entrada puede cambiar con fuerza el score de la clase objetivo.

Los mapas de gradientes suelen ser ruidosos.


In [ ]:
saliency_method = Saliency(model)

saliency_input = input_tensor.clone().detach().requires_grad_(True)

saliency_attr = saliency_method.attribute(
    saliency_input,
    target=target_class,
    abs=True,
)

saliency_map = (
    saliency_attr
    .squeeze(0)
    .max(dim=0)
    .values
    .detach()
    .cpu()
    .numpy()
)

plt.figure(figsize=(7, 5))
plt.imshow(saliency_map, cmap="hot")
plt.axis("off")
plt.title(f"Saliency — {target_name}")
plt.show()


## 6. Integrated Gradients

Integrated Gradients acumula gradientes a lo largo de un camino entre un **baseline** y la imagen real.

Esto reduce algunos problemas de usar únicamente el gradiente en el punto final.

Pero introduce una nueva decisión:

> ¿qué baseline representa una ausencia razonable de señal?


In [ ]:
N_STEPS = 32 #@param {type:"slider", min:8, max:128, step:8}

ig = IntegratedGradients(model)

ig_input = input_tensor.clone().detach().requires_grad_(True)
baseline = torch.zeros_like(ig_input)

ig_attr, convergence_delta = ig.attribute(
    ig_input,
    baselines=baseline,
    target=target_class,
    n_steps=N_STEPS,
    return_convergence_delta=True,
)

ig_map = (
    ig_attr
    .squeeze(0)
    .abs()
    .sum(dim=0)
    .detach()
    .cpu()
    .numpy()
)

print("Convergence delta:", float(convergence_delta.detach().cpu().reshape(-1)[0]))

plt.figure(figsize=(7, 5))
plt.imshow(ig_map, cmap="hot")
plt.axis("off")
plt.title(f"Integrated Gradients — {target_name}")
plt.show()


## 7. Superponer la atribución sobre la imagen

Una visualización más intuitiva ayuda a interpretar dónde se concentra la atribución.

Pero una visualización atractiva no es una métrica de calidad.


In [ ]:
def normalize_map(values):
    values = values - values.min()
    max_value = values.max()
    return values / max_value if max_value > 0 else values

ig_norm = normalize_map(ig_map)

plt.figure(figsize=(7, 5))
plt.imshow(image)
plt.imshow(ig_norm, cmap="hot", alpha=0.45)
plt.axis("off")
plt.title(f"Integrated Gradients superpuesto — {target_name}")
plt.show()


# 8. Comparar métodos y targets

Prueba:

- `TARGET_RANK = 1`;
- otra clase del top-5;
- `N_STEPS = 8`, `32`, `64`.

Observa:

- qué regiones permanecen;
- cuáles cambian;
- cuánto ruido aparece;
- cómo cambia el `convergence_delta`.


# Límites de saliency

Un mapa de atribución:

- depende de la clase objetivo;
- depende del método;
- puede depender del baseline;
- puede ser visualmente inestable;
- no demuestra causalidad;
- no demuestra que el modelo “entienda” el objeto;
- no es una garantía de robustez.

> Una explicación visual también necesita validación.


# Conclusiones de P04B

- Las explicaciones dependen del tipo de entrada y del modelo.
- Saliency e Integrated Gradients responden a preguntas relacionadas, pero no idénticas.
- El target y el baseline forman parte de la explicación.
- Una heatmap no es una prueba causal ni una descripción completa del razonamiento interno.
- La plausibilidad visual no basta para demostrar fidelidad.
